# FIFA World Cup 2026 — Prediction Model V11
**4-Layer Stacked Ensemble · 12 Features · Full Match Predictions · Team-Event What-Ifs**

| Layer | Model | Role |
|-------|-------|------|
| L1 | XGBoost + RF + ExtraTrees + LightGBM + MLP | Base learners |
| L2 | Logistic Regression + LightGBM | Meta-learners on OOF probabilities |
| L3 | MLP blender | Blend L2 outputs + passthrough |
| L4 | Sigmoid calibration | Calibrate probabilities |
| + | Specialist competitive model | 70/30 blend at inference |

**New in this version:** one consolidated 1,000-run Monte Carlo (the four 250-run parts each reset the
counters, so only the last 250 runs were ever reported), and a **team-events** engine for
"what if something happens to a team?" scenarios (section 19, 23).

In [ ]:
# ── 1  Imports ────────────────────────────────────────────────────────────────
import re, copy, warnings
from itertools import combinations
from collections import defaultdict
import numpy as np
import pandas as pd
import requests
from bs4 import BeautifulSoup
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from tqdm import tqdm
from sklearn.calibration import CalibratedClassifierCV
from sklearn.ensemble import (RandomForestClassifier, ExtraTreesClassifier,
                               GradientBoostingClassifier)
from sklearn.linear_model   import LogisticRegression
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing  import StandardScaler
from sklearn.pipeline       import Pipeline
from sklearn.metrics        import accuracy_score, classification_report
from sklearn.model_selection import StratifiedKFold
from sklearn.base           import BaseEstimator, ClassifierMixin
from xgboost import XGBClassifier

try:
    import lightgbm as lgb
    LGBM_OK = True
except ImportError:
    LGBM_OK = False
    print('LightGBM not found — GradientBoosting will be used as fallback')

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
np.random.seed(42)
HEADERS = {'User-Agent': 'Mozilla/5.0'}
print(f'Imports OK  |  LightGBM: {LGBM_OK}')

In [ ]:
# ── 2  Load data ──────────────────────────────────────────────────────────────
# Put the two CSVs in ./data (see data/README.md for where to get them).
from pathlib import Path
DATA_DIR = Path('data')

international = pd.read_csv(DATA_DIR / 'international_football_2018_present.csv', low_memory=False)
print(f'international   : {international.shape}')


In [ ]:
# ── 3  Name cleaning ──────────────────────────────────────────────────────────

HOME_NATION_SUFFIXES = {'eng':'England','wls':'Wales','sct':'Scotland','nir':'Northern Ireland'}

def build_name_map(teams):
    name_map = {}
    remaining = list(teams)
    for name in teams:                                    # prefix: 2 lower + uppercase
        m = re.match(r'^([a-z]{2})([A-Z].+)$', name)
        if m: name_map[name] = m.group(2).strip()
    for name in teams:                                    # home-nation 3-char suffixes
        if name in name_map: continue
        for sfx, country in HOME_NATION_SUFFIXES.items():
            if name == country + sfx: name_map[name] = country
    rset = set(n for n in remaining if n not in name_map)
    for name in remaining:                               # 2-char lowercase suffix
        if name in name_map: continue
        if len(name) >= 4 and name[-2:].islower():
            body = name[:-2]
            if body in rset: name_map[name] = body
    return name_map

NAME_MAP = build_name_map(international['team'].unique().tolist())
international['team']     = international['team'].map(lambda x: NAME_MAP.get(x, x))
international['opponent'] = international['opponent'].map(lambda x: NAME_MAP.get(x, x))
print(f'Dirty names fixed: {len(NAME_MAP)}')


In [ ]:
# ── 4  Elo scrape / fallback ──────────────────────────────────────────────────

ELO_NAME_MAP = {
    'Czech Republic':'Czechia', 'Bosnia':'Bosnia-Herzegovina',
    "Ivory Coast":"Côte d'Ivoire", 'Turkey':'Türkiye', 'Iran':'IR Iran',
    'Curacao':'Curaçao', 'DR Congo':'Congo DR',
    'Cape Verde Islands':'Cape Verde', 'South Korea':'Korea Republic',
    'USA':'United States',
}

ELO_FALLBACK = {
    'Mexico':1842,'South Africa':1621,'Korea Republic':1802,'Czechia':1764,
    'Canada':1766,'Bosnia-Herzegovina':1762,'Qatar':1607,'Switzerland':1879,
    'Brazil':2085,'Morocco':1934,'Haiti':1468,'Scotland':1797,
    'United States':1801,'Paraguay':1712,'Australia':1782,'Türkiye':1812,
    'Germany':2029,'Curaçao':1580,"Côte d'Ivoire":1831,'Ecuador':1757,
    'Netherlands':2008,'Japan':1907,'Sweden':1820,'Tunisia':1759,
    'Belgium':1960,'Egypt':1701,'IR Iran':1793,'New Zealand':1583,
    'Spain':2100,'Cape Verde':1678,'Saudi Arabia':1709,'Uruguay':1906,
    'France':2105,'Senegal':1848,'Iraq':1655,'Norway':1851,
    'Argentina':2142,'Algeria':1784,'Austria':1826,'Jordan':1627,
    'Portugal':2073,'Congo DR':1693,'Uzbekistan':1698,'Colombia':1840,
    'England':2002,'Croatia':1871,'Ghana':1705,'Panama':1634,
}
DEFAULT_ELO = 1500

def scrape_elo_ratings():
    try:
        resp = requests.get('https://www.eloratings.net/', headers=HEADERS, timeout=15)
        resp.raise_for_status()
        soup = BeautifulSoup(resp.text, 'html.parser')
        elo_dict = {}
        for row in soup.select('tr'):
            cols = row.find_all('td')
            if len(cols) >= 3:
                try:
                    name   = ELO_NAME_MAP.get(cols[1].get_text(strip=True), cols[1].get_text(strip=True))
                    rating = int(cols[2].get_text(strip=True).replace(',',''))
                    elo_dict[name] = rating
                except ValueError:
                    pass
        if len(elo_dict) > 50:
            print(f'✓ Scraped {len(elo_dict)} Elo ratings from eloratings.net')
            return elo_dict
        print('⚠ Sparse scrape — using hardcoded fallback')
        return ELO_FALLBACK
    except Exception as e:
        print(f'⚠ Elo scrape failed ({e}) — using hardcoded fallback')
        return ELO_FALLBACK

TEAM_ELO = scrape_elo_ratings()

international['date']         = pd.to_datetime(international['date'])
international                 = international.sort_values('date').reset_index(drop=True)
international['team_elo']     = international['team'].map(TEAM_ELO).fillna(DEFAULT_ELO)
international['opponent_elo'] = international['opponent'].map(TEAM_ELO).fillna(DEFAULT_ELO)

elo_df = pd.DataFrame(list(TEAM_ELO.items()), columns=['Team','Elo']).sort_values('Elo', ascending=False)
print('\nTop 10 by Elo:')
print(elo_df.head(10).to_string(index=False))


In [ ]:
# ── 5  Supporting lookups ─────────────────────────────────────────────────────
FIFA_RANK = {
    'Argentina':1,'France':2,'England':3,'Spain':4,'Brazil':5,
    'Portugal':6,'Netherlands':7,'Belgium':8,'Germany':9,'Uruguay':10,
    'Colombia':11,'Morocco':12,'Japan':13,'Croatia':14,'United States':15,
    'Mexico':16,'Switzerland':17,'Senegal':18,'Ecuador':21,'Austria':22,
    'Norway':23,'Australia':24,'Korea Republic':25,'Czechia':26,'Sweden':27,
    'IR Iran':28,'Türkiye':29,'Algeria':31,'Tunisia':32,'Paraguay':33,
    'Canada':35,'Scotland':36,'Egypt':37,'Saudi Arabia':38,'Ghana':40,
    "Côte d'Ivoire":42,'South Africa':43,'Uzbekistan':44,'Iraq':45,
    'Jordan':46,'Bosnia-Herzegovina':48,'Congo DR':50,'Qatar':51,
    'New Zealand':52,'Panama':53,'Haiti':54,'Cape Verde':55,'Curaçao':56,
}
DEFAULT_RANK = 80

CONFEDERATION = {
    'Argentina':'CONMEBOL','Brazil':'CONMEBOL','Uruguay':'CONMEBOL','Colombia':'CONMEBOL',
    'Ecuador':'CONMEBOL','Paraguay':'CONMEBOL',
    'France':'UEFA','England':'UEFA','Spain':'UEFA','Germany':'UEFA','Netherlands':'UEFA',
    'Portugal':'UEFA','Belgium':'UEFA','Croatia':'UEFA','Switzerland':'UEFA','Sweden':'UEFA',
    'Austria':'UEFA','Norway':'UEFA','Scotland':'UEFA','Czechia':'UEFA','Bosnia-Herzegovina':'UEFA',
    'United States':'CONCACAF','Mexico':'CONCACAF','Canada':'CONCACAF',
    'Panama':'CONCACAF','Haiti':'CONCACAF','Curaçao':'CONCACAF',
    'Japan':'AFC','Korea Republic':'AFC','Australia':'AFC','IR Iran':'AFC',
    'Saudi Arabia':'AFC','Qatar':'AFC','Iraq':'AFC','Jordan':'AFC','Uzbekistan':'AFC',
    'Morocco':'CAF','Senegal':'CAF','Egypt':'CAF','Ghana':'CAF',
    "Côte d'Ivoire":'CAF','Algeria':'CAF','Tunisia':'CAF','South Africa':'CAF',
    'Congo DR':'CAF','Cape Verde':'CAF','New Zealand':'OFC',
}
DEFAULT_CONF = 'OTHER'

# ── H2H win-rate from historical data ────────────────────────────────────────
def build_h2h(df):
    result_col = 'match_result' if 'match_result' in df.columns else 'result'
    label_map  = {'Win':1,'win':1,'Loss':0,'loss':0,'Draw':0.5,'draw':0.5}
    df2 = df.copy()
    df2['_res'] = df2[result_col].map(label_map).fillna(0.5)
    h2h = defaultdict(list)
    for _, row in df2.iterrows():
        key = tuple(sorted([row['team'], row['opponent']]))
        val = row['_res'] if row['team'] == key[0] else 1 - row['_res']
        h2h[key].append(val)
    return {k: round(np.mean(v), 4) for k, v in h2h.items()}

H2H = build_h2h(international)

def get_h2h(team_a, team_b):
    key = tuple(sorted([team_a, team_b]))
    wr  = H2H.get(key, 0.5)
    return wr if team_a == key[0] else 1 - wr

# ── Days since last match ─────────────────────────────────────────────────────
LAST_MATCH_DATE = {row['team']: row['date']
                   for _, row in international.sort_values('date').iterrows()}
REFERENCE_DATE  = pd.Timestamp('2026-06-11')

def days_since_last(team):
    d = LAST_MATCH_DATE.get(team)
    return max(0, (REFERENCE_DATE - d).days) if d is not None else 30

print(f'H2H pairs: {len(H2H):,}  |  FIFA ranks: {len(FIFA_RANK)}  |  Conf map: {len(CONFEDERATION)}')

In [ ]:
# ── 6  Form + xG ─────────────────────────────────────────────────────────────
def compute_recent_form(df, n=5):
    result_col = 'match_result' if 'match_result' in df.columns else 'result'
    label_map  = {'Win':1,'win':1,'Loss':0,'loss':0,'Draw':0.5,'draw':0.5}
    df2 = df.copy()
    df2['_res'] = df2[result_col].map(label_map).fillna(0.5)
    form = {}
    history = defaultdict(list)
    for idx, row in df2.sort_values('date').iterrows():
        hist = history[row['team']]
        form[idx] = sum(hist[-n:]) / n if len(hist) >= n else (sum(hist)/len(hist) if hist else 0.5)
        history[row['team']].append(row['_res'])
    return pd.Series(form)

# xG: try FBRef scrape, fall back to Elo-based proxy
def scrape_national_xg():
    try:
        resp = requests.get('https://fbref.com/en/comps/1/shooting/World-Cup-Stats',
                            headers=HEADERS, timeout=20)
        resp.raise_for_status()
        soup  = BeautifulSoup(resp.text, 'html.parser')
        table = soup.find('table', {'id': re.compile(r'stats_shooting')})
        xg_dict = {}
        if table:
            for row in table.select('tbody tr'):
                try:
                    team = row.find('td',{'data-stat':'team'}).get_text(strip=True)
                    xg   = float(row.find('td',{'data-stat':'xg'}).get_text(strip=True))
                    mp   = int(row.find('td',{'data-stat':'games'}).get_text(strip=True)) or 1
                    xg_dict[team] = round(xg / mp, 3)
                except (AttributeError, ValueError, ZeroDivisionError):
                    pass
        if len(xg_dict) > 10:
            print(f'✓ Scraped xG for {len(xg_dict)} teams')
            return xg_dict
        print('⚠ xG scrape sparse — using Elo-based proxy')
        return {}
    except Exception as e:
        print(f'⚠ xG scrape failed ({e}) — using Elo-based proxy')
        return {}

TEAM_XG_MAP = scrape_national_xg()

print('Computing rolling form...')
international['team_form'] = compute_recent_form(international, n=5)
# opponent form: flip the dataframe perspective
opp_df = international.rename(columns={'team':'opponent','opponent':'team'})
international['opponent_form'] = compute_recent_form(opp_df, n=5).reindex(international.index).fillna(0.5)

# Elo-based xG proxy (used when scraped xG is unavailable)
def elo_xg(team, elo_a, elo_b, side='team'):
    base = TEAM_XG_MAP.get(team, None)
    sign = 1 if side == 'team' else -1
    return round(max(0.3, (base if base else 1.25) + sign * (elo_a - elo_b) / 900), 3)

international['team_xg'] = international.apply(
    lambda r: elo_xg(r['team'], r['team_elo'], r['opponent_elo'], 'team'), axis=1)
international['opponent_xg'] = international.apply(
    lambda r: elo_xg(r['opponent'], r['team_elo'], r['opponent_elo'], 'opp'), axis=1)

print(f'Form & xG computed. Dataset: {international.shape}')

In [ ]:
# ── 7  Feature engineering ────────────────────────────────────────────────────
dataset = international.copy()

dataset['elo_diff']   = dataset['team_elo'] - dataset['opponent_elo']
dataset['value_diff'] = (pd.to_numeric(dataset['squad_value'],     errors='coerce') -
                          pd.to_numeric(dataset['opp_squad_value'], errors='coerce'))
dataset['xg_diff']    = dataset['team_xg'] - dataset['opponent_xg']

AGE_COLS     = ['team_avg_age','team_age']
OPP_AGE_COLS = ['opp_team_avg_age','opponent_avg_age','opponent_age','opp_team_age']
tc = next((c for c in AGE_COLS     if c in dataset.columns), None)
oc = next((c for c in OPP_AGE_COLS if c in dataset.columns), None)
dataset['age_diff'] = (
    pd.to_numeric(dataset[tc], errors='coerce') - pd.to_numeric(dataset[oc], errors='coerce')
    if tc and oc else 0.0)
dataset['form_diff'] = dataset['team_form'] - dataset['opponent_form']

# FIFA ranking diff (lower number = better → opponent_rank - team_rank is positive when team is better)
dataset['fifa_rank_diff'] = (
    dataset['opponent'].map(lambda t: FIFA_RANK.get(t, DEFAULT_RANK)) -
    dataset['team'].map(lambda t: FIFA_RANK.get(t, DEFAULT_RANK))
)
dataset['h2h_winrate'] = dataset.apply(
    lambda r: get_h2h(r['team'], r['opponent']), axis=1)
dataset['same_conf'] = (
    dataset['team'].map(lambda t: CONFEDERATION.get(t, DEFAULT_CONF)) ==
    dataset['opponent'].map(lambda t: CONFEDERATION.get(t, DEFAULT_CONF))
).astype(int)

def tournament_weight(row):
    t = str(row.get('tournament','') or '').lower()
    if any(k in t for k in ['world cup','fifa world cup']): return 2
    if any(k in t for k in ['qualifier','nations league','copa','euro','gold cup','africa cup']): return 1
    return 0

dataset['tournament_type'] = (
    dataset.apply(tournament_weight, axis=1)
    if 'tournament' in dataset.columns else 0
)
dataset['days_rest_diff'] = (
    dataset['team'].map(days_since_last) - dataset['opponent'].map(days_since_last))
dataset['home_adv'] = (
    (~dataset['neutral'].astype(bool)).astype(int)
    if 'neutral' in dataset.columns else 0
)

FEATURES = [
    'elo_diff','value_diff','xg_diff','age_diff','form_diff',    # core 5
    'fifa_rank_diff','h2h_winrate','same_conf',                   # rankings/rivalry 3
    'tournament_type','days_rest_diff','home_adv',                # context 3
    'team_xg',                                                    # absolute xG 1
]

for col in FEATURES:
    dataset[col] = pd.to_numeric(dataset[col], errors='coerce')
    dataset[col] = dataset[col].fillna(dataset[col].mean()).fillna(0)

label_map  = {'Loss':0,'Draw':1,'Win':2,'loss':0,'draw':1,'win':2}
result_col = 'match_result' if 'match_result' in dataset.columns else 'result'
dataset['target'] = dataset[result_col].map(label_map).fillna(1).astype(int)

latest_stats = dataset.sort_values('date').groupby('team').last().reset_index()

print(f'Features ({len(FEATURES)}): {FEATURES}')
print('\nTarget distribution:')
print(dataset['target'].value_counts().rename({0:'Loss',1:'Draw',2:'Win'}))

In [ ]:
# ── 8  Train/val split + sample weights ───────────────────────────────────────
def compute_sample_weights(df):
    """Linear recency weight (1→3) × competitive bonus (×1.5 for non-friendlies)."""
    min_d  = df['date'].min().timestamp()
    max_d  = df['date'].max().timestamp()
    rng    = max(max_d - min_d, 1)
    time_w = 1 + 2 * (df['date'].map(lambda d: d.timestamp()) - min_d) / rng
    comp_w = df['tournament_type'].map(lambda t: 1.5 if t > 0 else 1.0)
    w = time_w * comp_w
    return (w / w.mean()).values

train_df   = dataset[dataset['date'] < '2026-01-01'].copy()
val_df     = dataset[(dataset['date'] >= '2023-01-01') & (dataset['date'] < '2026-01-01')].copy()
train_comp = train_df[train_df['tournament_type'] > 0]

X_train, y_train = train_df[FEATURES].values, train_df['target'].values
X_val,   y_val   = val_df[FEATURES].values,   val_df['target'].values
W_train          = compute_sample_weights(train_df)

X_comp = train_comp[FEATURES].values
y_comp = train_comp['target'].values
W_comp = compute_sample_weights(train_comp) if len(train_comp) > 0 else np.ones(1)

print(f'Train (all)      : {len(X_train):,}')
print(f'  ├ competitive  : {len(X_comp):,}')
print(f'Val (2023–2025)  : {len(X_val):,}')
print(f'Weight range     : {W_train.min():.2f} – {W_train.max():.2f}')

In [ ]:
# ── 9  L1 estimators ──────────────────────────────────────────────────────────
xgb_l1 = XGBClassifier(
    n_estimators=500, max_depth=5, learning_rate=0.03,
    subsample=0.8, colsample_bytree=0.8,
    objective='multi:softprob', eval_metric='mlogloss',
    random_state=42, verbosity=0)

rf_l1  = RandomForestClassifier(n_estimators=400, max_depth=10, n_jobs=-1, random_state=42)
et_l1  = ExtraTreesClassifier( n_estimators=400, max_depth=10, n_jobs=-1, random_state=42)

lgbm_l1 = (
    lgb.LGBMClassifier(n_estimators=400, learning_rate=0.03, max_depth=5,
                       num_leaves=31, objective='multiclass', random_state=42, verbose=-1)
    if LGBM_OK else
    GradientBoostingClassifier(n_estimators=300, learning_rate=0.03, max_depth=5, random_state=42)
)

mlp_l1 = Pipeline([
    ('scaler', StandardScaler()),
    ('mlp', MLPClassifier(hidden_layer_sizes=(64,32), activation='relu',
                          max_iter=300, alpha=0.005, random_state=42)),
])

L1_ESTIMATORS = [('xgb',xgb_l1),('rf',rf_l1),('et',et_l1),('lgbm',lgbm_l1),('mlp',mlp_l1)]
print(f'{len(L1_ESTIMATORS)} L1 estimators defined.')

In [ ]:
# ── 10  OOF generation ────────────────────────────────────────────────────────

def build_oof_features(estimators, X, y, w, n_splits=5):
    """5-fold stratified OOF probabilities for each L1 estimator."""
    n_classes = len(np.unique(y))
    oof = np.zeros((len(X), len(estimators) * n_classes))
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)

    for fold, (tr_idx, va_idx) in enumerate(skf.split(X, y)):
        X_tr, X_va = X[tr_idx], X[va_idx]
        y_tr        = y[tr_idx]
        w_tr        = w[tr_idx]
        for j, (name, est) in enumerate(estimators):
            est_c = copy.deepcopy(est)
            try:
                if name in ['xgb','rf','et','lgbm']:
                    est_c.fit(X_tr, y_tr, sample_weight=w_tr)
                else:
                    est_c.fit(X_tr, y_tr)   # MLP pipeline
            except TypeError:
                est_c.fit(X_tr, y_tr)
            proba = est_c.predict_proba(X_va)
            if proba.shape[1] < n_classes:   # handle missing classes in fold
                tmp = np.zeros((len(X_va), n_classes))
                for ci, c in enumerate(est_c.classes_): tmp[:, c] = proba[:, ci]
                proba = tmp
            oof[va_idx, j*n_classes:(j+1)*n_classes] = proba
        print(f'  Fold {fold+1}/{n_splits} done')
    return oof

print('Building OOF features (5-fold)...')
OOF_TRAIN      = build_oof_features(L1_ESTIMATORS, X_train, y_train, W_train)
OOF_TRAIN_FULL = np.hstack([OOF_TRAIN, X_train])
print(f'OOF shape: {OOF_TRAIN_FULL.shape}')

In [ ]:
# ── 11  Train L1 on full data ─────────────────────────────────────────────────
# ── Fit L1 on full training set ──────────────────────────────────────────────
print('Training L1 estimators on full training data...')
L1_FITTED = []
for name, est in L1_ESTIMATORS:
    est_f = copy.deepcopy(est)
    try:
        if name in ['xgb','rf','et','lgbm']:
            est_f.fit(X_train, y_train, sample_weight=W_train)
        else:
            est_f.fit(X_train, y_train)
    except TypeError:
        est_f.fit(X_train, y_train)
    L1_FITTED.append((name, est_f))
    print(f'  ✓ {name}')

def l1_predict_proba(X):
    n_classes = 3
    out = np.zeros((len(X), len(L1_FITTED) * n_classes))
    for j, (name, est) in enumerate(L1_FITTED):
        p = est.predict_proba(X)
        if p.shape[1] < n_classes:
            tmp = np.zeros((len(X), n_classes))
            for ci, c in enumerate(est.classes_): tmp[:, c] = p[:, ci]
            p = tmp
        out[:, j*n_classes:(j+1)*n_classes] = p
    return out

# ── L2: meta-learners on OOF ──────────────────────────────────────────────────
print('Training L2 meta-learners...')
lr_l2 = LogisticRegression(max_iter=4000, solver='lbfgs')
lr_l2.fit(OOF_TRAIN_FULL, y_train, sample_weight=W_train)

lgbm_l2 = (
    lgb.LGBMClassifier(n_estimators=300, learning_rate=0.05, max_depth=4,
                       num_leaves=31, objective='multiclass', random_state=42, verbose=-1)
    if LGBM_OK else
    GradientBoostingClassifier(n_estimators=300, learning_rate=0.05, max_depth=4, random_state=42)
)
lgbm_l2.fit(OOF_TRAIN_FULL, y_train, sample_weight=W_train)

L2_TRAIN = np.hstack([
    lr_l2.predict_proba(OOF_TRAIN_FULL),
    lgbm_l2.predict_proba(OOF_TRAIN_FULL),
    OOF_TRAIN_FULL,
])
print(f'L2 output shape: {L2_TRAIN.shape}')

# ── L3: MLP blender ───────────────────────────────────────────────────────────
print('Training L3 MLP blender...')
mlp_l3 = Pipeline([
    ('scaler', StandardScaler()),
    ('mlp', MLPClassifier(hidden_layer_sizes=(64,32), activation='relu',
                          solver='adam', max_iter=300, alpha=0.005,
                          early_stopping=True, validation_fraction=0.1, random_state=42)),
])
mlp_l3.fit(L2_TRAIN, y_train)
print('L3 trained.')

# ── L4: calibration wrapper ───────────────────────────────────────────────────
class FourLayerStack(ClassifierMixin, BaseEstimator):
    """Wraps L1→L2→L3 as a single sklearn estimator for calibration."""
    _estimator_type = 'classifier'
    classes_ = np.array([0,1,2])

    def fit(self, X, y, sample_weight=None):
        self.classes_ = np.unique(y)
        return self

    def predict_proba(self, X):
        oof_x  = np.hstack([l1_predict_proba(X), X])
        l2_out = np.hstack([lr_l2.predict_proba(oof_x),
                            lgbm_l2.predict_proba(oof_x), oof_x])
        return mlp_l3.predict_proba(l2_out)

    def predict(self, X):
        return np.argmax(self.predict_proba(X), axis=1)

# NOTE: FourLayerStack.fit() is a no-op (L1-L3 are already fitted above), so the sigmoid
# calibrators are fitted on the stack's in-sample outputs. See README → Known limitations.
print('Calibrating L4 (sigmoid, 3-fold)...')
raw_stack = FourLayerStack()
raw_stack.fit(X_train, y_train)

cal_model = CalibratedClassifierCV(
    estimator=raw_stack,
    method='sigmoid',
    cv=3
)

cal_model.fit(X_train, y_train)

model = cal_model

print('L4 calibration done.')

# ── Specialist competitive model ──────────────────────────────────────────────
print('Training specialist competitive model...')
spec_xgb = XGBClassifier(
    n_estimators=400, max_depth=4, learning_rate=0.03,
    subsample=0.8, colsample_bytree=0.8,
    objective='multi:softprob', eval_metric='mlogloss',
    random_state=42, verbosity=0)

if len(X_comp) > 0 and len(np.unique(y_comp)) == 3:

    spec_xgb.fit(X_comp, y_comp, sample_weight=W_comp)

    spec_model = CalibratedClassifierCV(
        estimator=spec_xgb,
        method='sigmoid',
        cv=3
    )

    spec_model.fit(X_comp, y_comp)

    print('Specialist model trained on competitive matches.')

else:

    spec_model = cal_model
    print('Not enough competitive data — using full stack as specialist.')

# ── Blended inference function ────────────────────────────────────────────────
def blend_predict_proba(X, alpha=0.7):
    """70% full stack + 30% specialist competitive model."""
    return alpha * model.predict_proba(X) + (1 - alpha) * spec_model.predict_proba(X)

# ── Validation report ─────────────────────────────────────────────────────────
val_preds = np.argmax(blend_predict_proba(X_val), axis=1)
print(f'\nBlended accuracy (IN-SAMPLE: val window is part of train): {accuracy_score(y_val, val_preds):.4f}')
print(classification_report(y_val, val_preds, target_names=['Loss','Draw','Win'], zero_division=0))

In [ ]:
# ── 12  Prediction engine ─────────────────────────────────────────────────────

def get_features_v10(team_a, team_b, tournament_type=2, neutral=True):
    """Build the 12-feature vector for a WC 2026 match."""
    def _row(t):
        r = latest_stats[latest_stats['team'] == t]
        return r.iloc[0] if not r.empty else None

    ra, rb = _row(team_a), _row(team_b)

    def _val(row):
        if row is None: return 0.0
        v = pd.to_numeric(row.get('squad_value', 0), errors='coerce')
        return float(v) if pd.notna(v) else 0.0

    def _age(row):
        if row is None: return 26.0
        for c in ['team_avg_age','team_age']:
            if c in row.index:
                v = pd.to_numeric(row[c], errors='coerce')
                if pd.notna(v): return float(v)
        return 26.0

    def _form(row):
        if row is None: return 0.5
        v = row.get('team_form', 0.5)
        return float(v) if pd.notna(v) else 0.5

    def _xg(row, team):
        if row is None: return TEAM_XG_MAP.get(team, 1.25)
        v = row.get('team_xg', TEAM_XG_MAP.get(team, 1.25))
        return float(v) if pd.notna(v) else 1.25

    ea  = TEAM_ELO.get(team_a, DEFAULT_ELO)
    eb  = TEAM_ELO.get(team_b, DEFAULT_ELO)

    return np.array([[
        ea - eb,
        _val(ra) - _val(rb),
        _xg(ra, team_a) - _xg(rb, team_b),
        _age(ra) - _age(rb),
        _form(ra) - _form(rb),
        FIFA_RANK.get(team_b, DEFAULT_RANK) - FIFA_RANK.get(team_a, DEFAULT_RANK),
        get_h2h(team_a, team_b),
        int(CONFEDERATION.get(team_a, DEFAULT_CONF) == CONFEDERATION.get(team_b, DEFAULT_CONF)),
        tournament_type,
        days_since_last(team_a) - days_since_last(team_b),
        0 if neutral else 1,
        _xg(ra, team_a),
    ]])


def predict_probs_v10(team_a, team_b, tournament_type=2, neutral=True):
    """Return [P(loss_A), P(draw), P(win_A)]."""
    return blend_predict_proba(get_features_v10(team_a, team_b, tournament_type, neutral))[0]


def predict_match(team_a, team_b, knockout=False, tournament_type=2, neutral=True):
    """
    Predict a single match.
    Returns: (result_label, goals_a, goals_b, winner, xg_a, xg_b)
    result_label: 'Win <team_a>' | 'Draw' | 'Win <team_b>'
    winner: team name or None (group-stage draw)
    """
    probs = predict_probs_v10(team_a, team_b, tournament_type, neutral)
    ea    = TEAM_ELO.get(team_a, DEFAULT_ELO)
    eb    = TEAM_ELO.get(team_b, DEFAULT_ELO)
    xg_a  = TEAM_XG_MAP.get(team_a, max(0.3, 1.25 + (ea-eb)/900))
    xg_b  = TEAM_XG_MAP.get(team_b, max(0.3, 1.25 - (ea-eb)/900))
    mu_a, mu_b = max(0.3, xg_a), max(0.3, xg_b)

    if knockout:
        # No draws allowed — redistribute draw prob proportionally
        p_w = probs[2] + probs[1] * probs[2] / (probs[0] + probs[2] + 1e-9)
        outcome = 'W' if np.random.random() < p_w else 'L'
    else:
        outcome = np.random.choice(['W','D','L'], p=probs[[2,1,0]])

    ga = int(np.random.poisson(mu_a))
    gb = int(np.random.poisson(mu_b))

    if outcome == 'W':
        ga = max(ga, gb + 1)
        label, winner = f'Win {team_a}', team_a
    elif outcome == 'L':
        gb = max(gb, ga + 1)
        label, winner = f'Win {team_b}', team_b
    else:
        ga = gb = min(ga, gb)
        label, winner = 'Draw', None

    return label, ga, gb, winner, round(mu_a,3), round(mu_b,3)


print('V11 prediction engine ready (12 features, 4-layer blend).')

In [ ]:
# ── 13  Groups ────────────────────────────────────────────────────────────────
GROUPS = {
    'A': ['Mexico',        'South Africa',      'Korea Republic',   'Czechia'],
    'B': ['Canada',        'Bosnia-Herzegovina','Qatar',            'Switzerland'],
    'C': ['Brazil',        'Morocco',           'Haiti',            'Scotland'],
    'D': ['United States', 'Paraguay',          'Australia',        'Türkiye'],
    'E': ['Germany',       'Curaçao',           "Côte d'Ivoire",    'Ecuador'],
    'F': ['Netherlands',   'Japan',             'Sweden',           'Tunisia'],
    'G': ['Belgium',       'Egypt',             'IR Iran',          'New Zealand'],
    'H': ['Spain',         'Cape Verde',        'Saudi Arabia',     'Uruguay'],
    'I': ['France',        'Senegal',           'Iraq',             'Norway'],
    'J': ['Argentina',     'Algeria',           'Austria',          'Jordan'],
    'K': ['Portugal',      'Congo DR',          'Uzbekistan',       'Colombia'],
    'L': ['England',       'Croatia',           'Ghana',            'Panama'],
}
missing = [t for g in GROUPS.values() for t in g if t not in TEAM_ELO]
print('Teams missing Elo:', missing if missing else 'None — all 48 teams covered ✓')

In [ ]:
# ── 14  Group stage ───────────────────────────────────────────────────────────
group_match_results = []
group_standings     = {}

for grp, teams in GROUPS.items():
    table = {t: {'Team':t,'Pts':0,'W':0,'D':0,'L':0,'GF':0,'GA':0,'GD':0} for t in teams}

    for t1, t2 in combinations(teams, 2):
        label, g1, g2, winner, xg_a, xg_b = predict_match(t1, t2, knockout=False)
        group_match_results.append({
            'Group':  grp,
            'Team A': t1,
            'Score':  f'{g1}–{g2}',
            'Team B': t2,
            'Result': label,
            'Winner': winner if winner else 'Draw',
            'xG A':   xg_a,
            'xG B':   xg_b,
        })
        table[t1]['GF'] += g1;  table[t1]['GA'] += g2
        table[t2]['GF'] += g2;  table[t2]['GA'] += g1
        if winner == t1:
            table[t1]['Pts'] += 3; table[t1]['W'] += 1; table[t2]['L'] += 1
        elif winner == t2:
            table[t2]['Pts'] += 3; table[t2]['W'] += 1; table[t1]['L'] += 1
        else:
            table[t1]['Pts'] += 1; table[t1]['D'] += 1
            table[t2]['Pts'] += 1; table[t2]['D'] += 1

    for t in table: table[t]['GD'] = table[t]['GF'] - table[t]['GA']
    group_standings[grp] = sorted(
        table.values(), key=lambda x: (x['Pts'], x['GD'], x['GF']), reverse=True)

group_matches_df = pd.DataFrame(group_match_results)

for grp in GROUPS:
    print(f'\n══ Group {grp} ══')
    display(group_matches_df[group_matches_df['Group'] == grp]
            [['Team A','Score','Team B','Result','Winner']].reset_index(drop=True))

In [ ]:
# ── 15  Group standings ───────────────────────────────────────────────────────

all_standings = []
for grp, rows in group_standings.items():
    df = pd.DataFrame(rows)[['Team','Pts','W','D','L','GF','GA','GD']]
    df.insert(0, 'Pos', range(1, len(df)+1))
    df['Group'] = grp
    all_standings.append(df)
    print(f'\n── Group {grp} ──')
    display(df[['Pos','Team','Pts','W','D','L','GF','GA','GD']].reset_index(drop=True))

full_standings = pd.concat(all_standings, ignore_index=True)

In [ ]:
# ── 16  Qualification ─────────────────────────────────────────────────────────

group_winners = {g: rows[0]['Team'] for g, rows in group_standings.items()}
group_runners = {g: rows[1]['Team'] for g, rows in group_standings.items()}
third_pool    = [{**rows[2], 'Group': g} for g, rows in group_standings.items()]
best_thirds   = sorted(third_pool, key=lambda x: (x['Pts'],x['GD'],x['GF']), reverse=True)[:8]

qualified_32 = (
    list(group_winners.values()) +
    list(group_runners.values()) +
    [t['Team'] for t in best_thirds]
)

qual_df = pd.DataFrame({
    'Team':      qualified_32,
    'Route':     ['Winner']*12 + ['Runner-Up']*12 + ['Best 3rd']*8,
    'Elo':       [TEAM_ELO.get(t, DEFAULT_ELO) for t in qualified_32],
    'FIFA Rank': [FIFA_RANK.get(t, DEFAULT_RANK) for t in qualified_32],
}).sort_values('Elo', ascending=False).reset_index(drop=True)

print(f'{len(qualified_32)} teams qualified:')
display(qual_df)


In [ ]:
# ── 17  Knockout bracket ──────────────────────────────────────────────────────
knockout_log = []

def run_ko_round(pairs, round_name):
    winners = []
    for tA, tB in pairs:
        label, gA, gB, winner, _, _ = predict_match(tA, tB, knockout=True)
        loser = tB if winner == tA else tA
        knockout_log.append({
            'Round':  round_name,
            'Team A': tA,
            'Score':  f'{gA}–{gB}',
            'Team B': tB,
            'Winner': winner,
            'Loser':  loser,
        })
        winners.append(winner)
    return winners

# Round of 32 bracket: group winners vs runners of adjacent groups
thirds = [t['Team'] for t in best_thirds]
BRACKET_R32 = [
    (group_winners['A'], group_runners['B']),
    (group_winners['B'], group_runners['A']),
    (group_winners['C'], group_runners['D']),
    (group_winners['D'], group_runners['C']),
    (group_winners['E'], group_runners['F']),
    (group_winners['F'], group_runners['E']),
    (group_winners['G'], group_runners['H']),
    (group_winners['H'], group_runners['G']),
    (group_winners['I'], group_runners['J']),
    (group_winners['J'], group_runners['I']),
    (group_winners['K'], group_runners['L']),
    (group_winners['L'], group_runners['K']),
    (thirds[0], thirds[1]),
    (thirds[2], thirds[3]),
    (thirds[4], thirds[5]),
    (thirds[6], thirds[7]),
]

r32_w = run_ko_round(BRACKET_R32, 'Round of 32')
print(f'R32 complete → {len(r32_w)} winners')

r16_w = run_ko_round([(r32_w[i], r32_w[i+1]) for i in range(0,16,2)], 'Round of 16')
print(f'R16 complete → {len(r16_w)} winners')

qf_w  = run_ko_round([(r16_w[i], r16_w[i+1]) for i in range(0,8,2)],  'Quarter-Finals')
print(f'QF  complete → {len(qf_w)} winners')

sf_w  = run_ko_round([(qf_w[i],  qf_w[i+1])  for i in range(0,4,2)],  'Semi-Finals')
print(f'SF  complete → {len(sf_w)} winners')

sf_losers = [r['Loser'] for r in knockout_log if r['Round'] == 'Semi-Finals']
run_ko_round([(sf_losers[0], sf_losers[1])], '3rd Place Play-off')
run_ko_round([(sf_w[0], sf_w[1])],           'Final')

knockout_df = pd.DataFrame(knockout_log)
champion    = knockout_df[knockout_df['Round'] == 'Final'].iloc[0]['Winner']
runner_up   = knockout_df[knockout_df['Round'] == 'Final'].iloc[0]['Loser']
third_place = knockout_df[knockout_df['Round'] == '3rd Place Play-off'].iloc[0]['Winner']

print(f'\n🏆  Champion  : {champion}')
print(f'🥈  Runner-Up : {runner_up}')
print(f'🥉  3rd Place : {third_place}')


In [ ]:
for rnd in [
    'Round of 32',
    'Round of 16',
    'Quarter-Finals',
    'Semi-Finals',
    '3rd Place Play-off',
    'Final'
]:
    print(f"\n{'═'*55}\n  {rnd}\n{'═'*55}")

    display(
        knockout_df[knockout_df['Round'] == rnd][
            ['Team A', 'Score', 'Team B', 'Winner']
        ].reset_index(drop=True)
    )

## Monte Carlo & team events

In [ ]:
# ── 19  Team events + Monte Carlo engine ────────────────────────────────────
# "Something happens to a team": a withdrawal / disqualification, or an Elo shock
# (key injury, coach sacked, travel chaos ...). Events are applied on top of the
# model's own probabilities, so no retraining is needed.
from functools import lru_cache
from dataclasses import dataclass
from typing import Optional

STAGES = ['R32', 'R16', 'QF', 'SF', 'Final', 'Champion']

@dataclass(frozen=True)
class TeamEvent:
    """kind='withdraw'  -> team forfeits (3-0 walkovers); optional `replacement` swaps a new team in.
       kind='elo_shock' -> team's strength shifts by `delta` Elo points (negative = weaker).
       knockout_only=True -> the event only kicks in from the Round of 32."""
    kind: str
    team: str
    delta: float = 0.0
    knockout_only: bool = False
    replacement: Optional[str] = None

@lru_cache(maxsize=None)
def base_probs(a, b):
    """Cached model probabilities [P(loss_a), P(draw), P(win_a)] — the model is only run once per pair."""
    return tuple(predict_probs_v10(a, b))

def tilt_probs(p, delta):
    """Shift team A's strength by `delta` Elo points, keeping P(draw) fixed."""
    if delta == 0:
        return np.asarray(p, dtype=float)
    p_draw = p[1]
    e = min(max(p[2] + 0.5 * p_draw, 1e-6), 1 - 1e-6)
    x = 400 * np.log10(e / (1 - e)) + delta
    e2 = 1 / (1 + 10 ** (-x / 400))
    p_win  = max(e2 - 0.5 * p_draw, 0.0)
    p_loss = max((1 - e2) - 0.5 * p_draw, 0.0)
    s = p_win + p_draw + p_loss
    return np.array([p_loss / s, p_draw / s, p_win / s])

def build_scenario(events=()):
    groups = {g: list(ts) for g, ts in GROUPS.items()}
    shock_group, shock_ko = defaultdict(float), defaultdict(float)
    out_group, out_ko = set(), set()

    def find(team):
        for g, ts in groups.items():
            if team in ts:
                return g, ts.index(team)
        return None

    for ev in events:
        loc = find(ev.team)
        if loc is None:
            raise ValueError(f"'{ev.team}' is not in the tournament")
        if ev.kind == 'elo_shock':
            if not ev.knockout_only:
                shock_group[ev.team] += ev.delta
            shock_ko[ev.team] += ev.delta
        elif ev.kind == 'withdraw':
            if ev.replacement:
                if ev.knockout_only:
                    raise ValueError('a replacement only makes sense before the group stage')
                if find(ev.replacement):
                    raise ValueError(f"replacement '{ev.replacement}' is already in the tournament")
                if ev.replacement not in TEAM_ELO:
                    print(f"⚠ no Elo for {ev.replacement} — using DEFAULT_ELO={DEFAULT_ELO}")
                g, i = loc
                groups[g][i] = ev.replacement
            else:
                if not ev.knockout_only:
                    out_group.add(ev.team)
                out_ko.add(ev.team)
        else:
            raise ValueError(f"unknown event kind '{ev.kind}'")
    teams = [t for ts in groups.values() for t in ts]
    return dict(groups=groups, teams=teams, shock_group=shock_group, shock_ko=shock_ko,
                out_group=out_group, out_ko=out_ko)

def _group_match(sc, a, b, rng):
    ao, bo = a in sc['out_group'], b in sc['out_group']
    if ao and bo: return 0, 0
    if ao:        return 0, 3          # walkover for b
    if bo:        return 3, 0          # walkover for a
    p = tilt_probs(base_probs(a, b), sc['shock_group'][a] - sc['shock_group'][b])
    outcome = rng.choice(3, p=p[[2, 1, 0]])          # 0 = a wins, 1 = draw, 2 = b wins
    ea, eb = TEAM_ELO.get(a, DEFAULT_ELO), TEAM_ELO.get(b, DEFAULT_ELO)
    ga = int(rng.poisson(max(0.3, TEAM_XG_MAP.get(a, 1.25 + (ea - eb) / 900))))
    gb = int(rng.poisson(max(0.3, TEAM_XG_MAP.get(b, 1.25 - (ea - eb) / 900))))
    if outcome == 0:   ga = max(ga, gb + 1)
    elif outcome == 2: gb = max(gb, ga + 1)
    else:              ga = gb = min(ga, gb)
    return ga, gb

def _knockout(sc, a, b, rng):
    ao, bo = a in sc['out_ko'], b in sc['out_ko']
    if ao and not bo: return b
    if bo and not ao: return a
    if ao and bo:     return a if rng.random() < 0.5 else b
    p  = tilt_probs(base_probs(a, b), sc['shock_ko'][a] - sc['shock_ko'][b])
    pw = p[2] + p[1] * p[2] / (p[0] + p[2] + 1e-9)       # draws redistributed (extra time / pens)
    return a if rng.random() < pw else b

_R32_PAIRS = [('A','B'),('B','A'),('C','D'),('D','C'),('E','F'),('F','E'),
              ('G','H'),('H','G'),('I','J'),('J','I'),('K','L'),('L','K')]

def simulate_once(sc, rng, counts):
    winners, runners, thirds = {}, {}, []
    for g, teams in sc['groups'].items():
        tab = {t: [0, 0, 0, rng.random()] for t in teams}          # pts, gd, gf, random tiebreak
        for t1, t2 in combinations(teams, 2):
            g1, g2 = _group_match(sc, t1, t2, rng)
            tab[t1][2] += g1; tab[t2][2] += g2
            tab[t1][1] += g1 - g2; tab[t2][1] += g2 - g1
            if g1 > g2:   tab[t1][0] += 3
            elif g2 > g1: tab[t2][0] += 3
            else:         tab[t1][0] += 1; tab[t2][0] += 1
        order = sorted(teams, key=lambda t: (-tab[t][0], -tab[t][1], -tab[t][2], tab[t][3]))
        winners[g], runners[g] = order[0], order[1]
        thirds.append((order[2], tab[order[2]]))
    thirds.sort(key=lambda x: (-x[1][0], -x[1][1], -x[1][2], x[1][3]))
    t3 = [t for t, _ in thirds[:8]]                 # best 8 third-placed teams (pts, GD, GF)

    r32 = [(winners[a], runners[b]) for a, b in _R32_PAIRS] + [(t3[i], t3[i+1]) for i in range(0, 8, 2)]
    for a, b in r32:
        counts[a][0] += 1; counts[b][0] += 1
    rnd = [_knockout(sc, a, b, rng) for a, b in r32]
    for stage in range(1, 6):
        for t in rnd:
            counts[t][stage] += 1
        if len(rnd) == 1:
            break
        rnd = [_knockout(sc, rnd[i], rnd[i+1], rng) for i in range(0, len(rnd), 2)]

def run_simulations(events=(), sims=1000, seed=42, show_progress=True):
    """Return a DataFrame: % of simulations in which each team reaches each stage."""
    sc = build_scenario(events)
    rng = np.random.default_rng(seed)
    counts = {t: np.zeros(6) for t in sc['teams']}
    for _ in (tqdm(range(sims), desc='Simulating') if show_progress else range(sims)):
        simulate_once(sc, rng, counts)
    df = pd.DataFrame(counts, index=STAGES).T / sims * 100
    df.index.name = 'Team'
    return df.sort_values(['Champion', 'Final', 'SF'], ascending=False).round(1)

def compare_scenarios(events, sims=1000, seed=42, top=15):
    """Baseline vs. 'something happened' — same seed, so the difference is the event, not noise."""
    base = run_simulations((), sims, seed, show_progress=False)
    scen = run_simulations(events, sims, seed, show_progress=False)
    out = scen[['Champion', 'Final', 'SF']].join(base[['Champion']].rename(columns={'Champion': 'Champion (base)'}), how='left')
    out['Δ Champion (pp)'] = (out['Champion'] - out['Champion (base)'].fillna(0)).round(1)
    return out.sort_values('Champion', ascending=False).head(top)


In [ ]:
# ── 20  Baseline Monte Carlo (no events) ─────────────────────────────────────
SIMS = 1000
mc_probs = run_simulations(events=(), sims=SIMS, seed=42)

# Keep the dict names the results cells below expect
champ_count  = {t: v / 100 * SIMS for t, v in mc_probs['Champion'].items() if v > 0}
runner_count = {t: (mc_probs.loc[t, 'Final'] - mc_probs.loc[t, 'Champion']) / 100 * SIMS
                for t in mc_probs.index if mc_probs.loc[t, 'Final'] > mc_probs.loc[t, 'Champion']}
sf_count     = {t: v / 100 * SIMS for t, v in mc_probs['SF'].items() if v > 0}
print(f'Done — {SIMS:,} tournaments simulated.')


In [ ]:
# ── 20  Results tables ────────────────────────────────────────────────────────
champion_probs = (
    pd.DataFrame({'Team':list(champ_count),
                  'Champion%': [v/SIMS*100 for v in champ_count.values()]})
    .sort_values('Champion%', ascending=False).reset_index(drop=True))

runner_probs = (
    pd.DataFrame({'Team':list(runner_count),
                  'Runner-Up%': [v/SIMS*100 for v in runner_count.values()]})
    .sort_values('Runner-Up%', ascending=False).reset_index(drop=True))

sf_probs = (
    pd.DataFrame({'Team':list(sf_count),
                  'Semi-Final%': [v/SIMS*100 for v in sf_count.values()]})
    .sort_values('Semi-Final%', ascending=False).reset_index(drop=True))

full_probs = (
    champion_probs
    .merge(runner_probs, on='Team', how='outer')
    .merge(sf_probs,     on='Team', how='outer')
    .fillna(0)
    .sort_values('Champion%', ascending=False)
    .reset_index(drop=True)
)
full_probs[['Champion%','Runner-Up%','Semi-Final%']] =full_probs[['Champion%','Runner-Up%','Semi-Final%']].round(1)

print(f'Top 20 — Monte Carlo ({SIMS:,} sims)')
display(full_probs.head(20))

In [ ]:
# ── 21  Summary ───────────────────────────────────────────────────────────────
print('=' * 62)
print('   2026 FIFA WORLD CUP — FULL PREDICTION SUMMARY  (V11)')
print('=' * 62)
print(f'\n🏆  Champion  : {champion}')
print(f'🥈  Runner-Up : {runner_up}')
print(f'🥉  3rd Place : {third_place}')
print()
print('Model architecture:')
print('  L1 : XGBoost + RF + ExtraTrees + LightGBM + MLP')
print('  L2 : Logistic Regression + LightGBM (meta-learners on OOF probs)')
print('  L3 : MLP blender on L2 outputs + passthrough features')
print('  L4 : Sigmoid calibration')
print('   +  Specialist competitive model (70/30 blend at inference)')
print(f'  Features ({len(FEATURES)}): {FEATURES}')
print()
print('── Group Stage: All 96 Matches ─────────────────────────')
display(group_matches_df[['Group','Team A','Score','Team B','Winner']])
print()
print('── Knockout Stage: All 31 Matches ──────────────────────')
display(knockout_df[['Round','Team A','Score','Team B','Winner']])
print()
print(f'── Monte Carlo Championship Odds (Top 15, {SIMS:,} sims) ──')
display(full_probs.head(15))

In [ ]:
# ── 22  Charts ────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(18, 8))

# Left: championship probability bar chart
top20c  = champion_probs.head(20).sort_values('Champion%')
colors  = ['#c0392b' if t == champion else '#2980b9' for t in top20c['Team']]
bars    = axes[0].barh(top20c['Team'], top20c['Champion%'], color=colors, edgecolor='white')
axes[0].bar_label(bars, fmt='%.1f%%', padding=3, fontsize=8)
axes[0].set_title('Championship Win Probability — Top 20', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Probability (%)')
axes[0].grid(axis='x', alpha=0.3)
axes[0].legend(
    handles=[mpatches.Patch(color='#c0392b', label=f'Predicted champion: {champion}')],
    fontsize=9)

# Right: heatmap of stage probabilities
heat = full_probs.head(20).set_index('Team')[['Champion%','Runner-Up%','Semi-Final%']]
im   = axes[1].imshow(heat.values, aspect='auto', cmap='YlOrRd')
axes[1].set_xticks(range(3))
axes[1].set_xticklabels(['Champion','Runner-Up','Semi-Final'], fontsize=10)
axes[1].set_yticks(range(len(heat)))
axes[1].set_yticklabels(heat.index, fontsize=9)
for i in range(len(heat)):
    for j in range(3):
        axes[1].text(j, i, f'{heat.values[i,j]:.1f}%', ha='center', va='center', fontsize=7)
axes[1].set_title('Knockout Stage Probabilities — Top 20', fontsize=12, fontweight='bold')
plt.colorbar(im, ax=axes[1], label='%')

plt.suptitle(
    f'FIFA World Cup 2026 — V11  (4-Layer Stack · 12 Features · {SIMS:,} MC sims)',
    fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('wc2026_predictions_v11.png', dpi=150, bbox_inches='tight')
plt.show()
print('Chart saved → wc2026_predictions_v11.png')

## What if something happens to a team?
Add one or more `TeamEvent`s and compare against the baseline (same random seed, so differences come from the event, not noise).

In [ ]:
# ── 23  What-if: something happens to a team ────────────────────────────────
# Edit EVENTS and re-run. Examples:
#   TeamEvent('elo_shock', 'France', delta=-100)                    # star striker injured
#   TeamEvent('elo_shock', 'Brazil', delta=-60, knockout_only=True) # fatigue / injuries before the R32
#   TeamEvent('withdraw',  'United States')                         # forfeits every match
#   TeamEvent('withdraw',  'Iraq', replacement='Italy')             # swapped for another team
EVENTS = [
    TeamEvent('elo_shock', 'France', delta=-100),
]

comparison = compare_scenarios(EVENTS, sims=SIMS, seed=42)
print('Scenario:', ', '.join(f"{e.kind}({e.team}{', ' + str(e.delta) if e.kind == 'elo_shock' else ''})" for e in EVENTS))
display(comparison)


## Export for the Rust simulator
Writes the model's pairwise win/draw/loss probabilities so the Rust port (`rust/`) can reuse the trained ensemble.

In [ ]:
# ── 24  Export the model's pairwise probabilities for the Rust simulator ────
def export_prob_matrix(path='wc2026_prob_matrix.csv'):
    teams = [t for ts in GROUPS.values() for t in ts]
    pairs = [(a, b) for a in teams for b in teams if a != b]
    X = np.vstack([get_features_v10(a, b) for a, b in pairs])
    P = blend_predict_proba(X)                         # one batched call, not 2,256 single ones
    pd.DataFrame({'team_a': [a for a, _ in pairs], 'team_b': [b for _, b in pairs],
                  'p_loss': P[:, 0], 'p_draw': P[:, 1], 'p_win': P[:, 2]}
                 ).to_csv(path, index=False, float_format='%.6f')
    print(f'Wrote {len(pairs):,} rows → {path}')

export_prob_matrix()
